# Cochleogram-ViT — Best Baseline + MixUp

Builds on the sweep's best config (`loss-p025`: single correction, weighted CE loss,
SOFTEN_POWER=0.25, no KAN, leak-free Subset) → **64.67** pooled (paper convention).

**One change vs that config: MixUp during training.** Each batch is linearly mixed with
a randomly permuted copy of itself:

    lam = Beta(alpha, alpha)
    x_mix = lam * x + (1 - lam) * x[perm]
    loss  = lam * CE(model(x_mix), y) + (1 - lam) * CE(model(x_mix), y[perm])

(Standard MixUp; works fine with weighted CE — each sample's class weight is preserved.)
Validation is computed on CLEAN (un-mixed) batches.

Uses the EXISTING precomputed cochleograms (no new preprocessing). Per-fold val
softmax probabilities are saved alongside the JSON so we can ensemble with sweep
results later.

`MIXUP_ALPHA` is a one-line knob — defaults to 0.4 (common sweet spot for image
classification). Try 0.2 (more aggressive) or 1.0 (uniform mixing) by editing cell 2.


In [1]:
# --- Imports + data setup ---
import os, json, time, copy, gc
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

from cochleogram_vit.models.vit import CochleogramViT

DATA_DIR      = '../data/processed/cochleograms'
METADATA_PATH = '../data/processed/metadata.csv'
RESULTS_PATH  = '../results/mixup.json'
PREDS_PATH    = '../results/mixup_preds.npz'

BATCH_SIZE    = 16
EPOCHS        = 30
LEARNING_RATE = 1e-4


class CochleogramDataset(Dataset):
    def __init__(self, data_dir, metadata_path):
        self.data_dir = data_dir
        self.metadata = pd.read_csv(metadata_path)
        self._viridis = plt.get_cmap('viridis')
    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        npy_path = os.path.join(self.data_dir, os.path.basename(row['npy_path']))
        coch = np.load(npy_path)
        rgb = self._viridis(coch)[:, :, :3].transpose(2, 0, 1)
        return torch.from_numpy(np.ascontiguousarray(rgb)).float(), int(row['label'])

dataset = CochleogramDataset(DATA_DIR, METADATA_PATH)
metadata = dataset.metadata.copy()
metadata['patient_id'] = metadata['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dataset size: {len(dataset)}  device: {device}')

gkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)  # was GroupKFold
FOLDS = list(gkf.split(metadata, metadata['label'].values, groups=metadata['patient_id'].values))
print(f'StratifiedGroupKFold: {len(FOLDS)} folds')

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)


Dataset size: 6898  device: cuda
StratifiedGroupKFold: 10 folds


In [2]:
# --- Config (the only experiment-specific cell) ---
SOFTEN_POWER = 0.25     # best from sweep (loss-p025 = 64.67)
MIXUP_ALPHA  = 0.4      # MixUp Beta(alpha, alpha) -- 0.2 aggressive, 0.4 standard, 1.0 uniform
MIXUP_PROB   = 1.0      # probability of applying MixUp per batch (1.0 = always)

print(f'SOFTEN_POWER={SOFTEN_POWER}  MIXUP_ALPHA={MIXUP_ALPHA}  MIXUP_PROB={MIXUP_PROB}')
print(f'EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}  LR={LEARNING_RATE}')


SOFTEN_POWER=0.25  MIXUP_ALPHA=0.4  MIXUP_PROB=1.0
EPOCHS=30  BATCH_SIZE=16  LR=0.0001


In [3]:
# --- Helpers ---
def softened_weights(softpow):
    raw = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=metadata['label'].values)
    w = raw ** softpow
    return w / w.sum() * len(w)


def lr_lambda(epoch):
    warmup = 4
    if epoch < warmup:
        return (epoch + 1) / warmup
    denom = EPOCHS - warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / denom)) if denom > 0 else 0.0


def paper_metrics(preds, labels):
    p = np.asarray(preds); y = np.asarray(labels)
    TP   = int(np.sum((y != 0) & (p == y)))
    FN   = int(np.sum((y != 0) & (p == 0)))
    FN_w = int(np.sum((y != 0) & (p != 0) & (p != y)))
    TN   = int(np.sum((y == 0) & (p == 0)))
    FP   = int(np.sum((y == 0) & (p != 0)))
    TP_b = TP + FN_w   # paper convention
    se = TP_b / (TP_b + FN + 1e-8)
    sp = TN / (TN + FP + 1e-8)
    return {'TP': TP, 'FN': FN, 'FN_wrong': FN_w, 'TN': TN, 'FP': FP,
            'se': float(se), 'sp': float(sp), 'score': float((se + sp) / 2)}


def mixup_batch(x, y, alpha):
    """Returns (x_mixed, y_a, y_b, lam) for a MixUp step."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1.0 - lam) * x[perm]
    return x_mix, y, y[perm], lam


@torch.no_grad()
def evaluate(model, loader):
    """Returns argmax preds, labels, and softmax probs (N, 4) for ensembling."""
    model.eval()
    preds, labels, probs = [], [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        p = torch.softmax(logits, dim=1)
        probs.append(p.cpu().numpy())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        labels.extend(y.numpy().tolist())
    return preds, labels, np.concatenate(probs, axis=0)


In [4]:
# --- 10-fold training with MixUp ---
cw = softened_weights(SOFTEN_POWER)
cw_t = torch.tensor(cw, dtype=torch.float).to(device)
print(f'class weights (^{SOFTEN_POWER}): {np.round(cw, 3).tolist()}')

fold_rows, pooled_preds, pooled_labels = [], [], []
per_fold_probs = {}     # for ensembling later
per_fold_labels = {}
per_fold_val_idx = {}

t_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLDS):
    torch.manual_seed(42 + fold); np.random.seed(42 + fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42 + fold)

    train_subset = Subset(dataset, train_idx)
    val_subset   = Subset(dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    model = CochleogramViT(
        image_size=128, patch_size=16, num_classes=4,
        dim=512, depth=6, heads=8, mlp_dim=1024, channels=3,
        dropout=0.3, emb_dropout=0.2,
    ).to(device)
    opt   = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    criterion = nn.CrossEntropyLoss(weight=cw_t)

    best_score = -1.0; best_state = None; best_epoch = 0
    for epoch in range(EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            if np.random.random() < MIXUP_PROB:
                x_mix, y_a, y_b, lam = mixup_batch(x, y, MIXUP_ALPHA)
            else:
                x_mix, y_a, y_b, lam = x, y, y, 1.0
            opt.zero_grad()
            out = model(x_mix)
            loss = lam * criterion(out, y_a) + (1.0 - lam) * criterion(out, y_b)
            loss.backward()
            opt.step()

        preds, labels, _ = evaluate(model, val_loader)
        m = paper_metrics(preds, labels)
        if m['score'] > best_score:
            best_score = m['score']
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
        sched.step()

    # final eval at best checkpoint -- capture softmax probs for ensembling
    model.load_state_dict(best_state)
    preds, labels, probs = evaluate(model, val_loader)
    m = paper_metrics(preds, labels)
    fold_rows.append({'fold': fold + 1, 'best_epoch': best_epoch, **m})
    pooled_preds.extend(preds); pooled_labels.extend(labels)
    per_fold_probs[f'fold{fold+1}_probs']  = probs.astype(np.float32)
    per_fold_labels[f'fold{fold+1}_labels'] = np.asarray(labels, dtype=np.int64)
    per_fold_val_idx[f'fold{fold+1}_val_idx'] = np.asarray(val_idx, dtype=np.int64)
    print(f'fold {fold+1:>2}: best_ep={best_epoch:>2}  Se={m["se"]*100:5.2f}  Sp={m["sp"]*100:5.2f}  Score={m["score"]*100:5.2f}')

    del model, opt, sched, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pooled = paper_metrics(pooled_preds, pooled_labels)
se_mean = float(np.mean([r['se']    for r in fold_rows]))
sp_mean = float(np.mean([r['sp']    for r in fold_rows]))
sc_mean = float(np.mean([r['score'] for r in fold_rows]))
sc_std  = float(np.std ([r['score'] for r in fold_rows]))
elapsed = time.time() - t_start

result = {
    'id': f'mixup-a{MIXUP_ALPHA}-p{MIXUP_PROB}-soft{SOFTEN_POWER}',
    'config': {'soften_power': SOFTEN_POWER, 'mixup_alpha': MIXUP_ALPHA, 'mixup_prob': MIXUP_PROB,
               'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE},
    'elapsed_sec': elapsed,
    'folds': fold_rows,
    'per_fold_mean': {'se': se_mean, 'sp': sp_mean, 'score': sc_mean, 'score_std': sc_std},
    'pooled_aggregate': pooled,
}
with open(RESULTS_PATH, 'w') as f:
    json.dump(result, f, indent=2, default=str)
np.savez(PREDS_PATH, **per_fold_probs, **per_fold_labels, **per_fold_val_idx)
print(f'\n-> {RESULTS_PATH}')
print(f'-> {PREDS_PATH}')
print(f'\nPER-FOLD MEAN: Se={se_mean*100:.2f}  Sp={sp_mean*100:.2f}  Score={sc_mean*100:.2f}  std={sc_std*100:.2f}')
print(f'POOLED:        Se={pooled["se"]*100:.2f}  Sp={pooled["sp"]*100:.2f}  Score={pooled["score"]*100:.2f}')
print(f'elapsed: {elapsed/60:.1f} min')


class weights (^0.25): [0.763, 0.902, 1.086, 1.249]
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  1: best_ep= 7  Se=65.58  Sp=58.95  Score=62.27
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  2: best_ep=26  Se=38.71  Sp=62.09  Score=50.40
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  3: best_ep=17  Se=52.41  Sp=69.59  Score=61.00
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  4: best_ep=26  Se=50.15  Sp=61.81  Score=55.98
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  5: best_ep= 4  Se=35.14  Sp=89.86  Score=62.50
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  6: best_ep= 4  Se=61.99  Sp=73.83  Score=67.91
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  7: best_ep= 4  Se=52.84  Sp=83.33  Score=68.09
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
fold  8: 

In [5]:
# --- Compare against sweep results ---
print(f"{'config':<30} {'per-fold Sc±std':<18} {'pooled':<8}")
print('-' * 60)

# this run
m = result['per_fold_mean']; p = result['pooled_aggregate']
print(f"{result['id']:<30} {m['score']*100:5.2f}±{m['score_std']*100:4.2f}        {p['score']*100:.2f}")

# sweep results for reference
sweep_path = '../results/sweep_overnight.json'
if os.path.exists(sweep_path):
    sweep = json.load(open(sweep_path))
    for r in sweep:
        if 'error' in r: continue
        m, p = r['per_fold_mean'], r['pooled_aggregate']
        print(f"{r['id']:<30} {m['score']*100:5.2f}±{m['score_std']*100:4.2f}        {p['score']*100:.2f}")


config                         per-fold Sc±std    pooled  
------------------------------------------------------------
mixup-a0.4-p1.0-soft0.25       61.84±6.06        62.26
loss-p05                       64.86±5.70        63.21
loss-p025                      65.25±5.49        64.67
sampler-p05                    64.27±4.82        64.31
kan-last-p05                   64.43±5.96        64.33
kan-first-p05                  64.75±5.20        63.54
